<div align="center">

# WeightsLab · Local Studio quickstart (notebook-driven experiment)

Run your **whole experiment from this notebook** while the Weights Studio UI runs
**independently** next to it — no `main.py`, no Docker, no tunnel.

</div>


## How this works

This notebook's **kernel is the backend**: when you call `wl.serve(serving_grpc=True)`
below, *this Python process* starts the gRPC backend and holds the live `model`,
dataframe and ledgers. The UI is a **separate, independent process**
(`weightslab start`) that connects to this kernel over `localhost:50051`.

- **Kernel (here)** — a normal local Python process. Full native access to your
  packages, GPU and filesystem — it *is* a real script, no sandbox.
- **UI (`weightslab start`)** — standalone. Launch it before or after this
  notebook; it shows a landing page until a backend appears, then connects.
- **Training** runs in a **background thread** (last section) so this notebook
  stays interactive while the UI streams the run live.

> Prerequisite (one terminal, once):
> ```bash
> pip install weightslab torch torchvision torchmetrics
> ```


In [ ]:
# Optional: install into this kernel if not already present.
# %pip install weightslab torch torchvision torchmetrics


## 1 · Imports

In [ ]:
import os
import threading
import logging

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.examples.utils.baseline_models.pytorch.models import FashionCNN as CNN
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2 · Experiment directory

`root_log_dir` is where this run's checkpoints, signal history, dataframes and the
in-browser `notebook.ipynb` live. Point the **UI** at the *same* directory so the
two share one experiment:

```bash
weightslab start ./exp/mnist_mode_c      # in a separate terminal
```

`weightslab start [DIR]` also exports `WEIGHTSLAB_ROOT_LOG_DIR`; we honor it here so
the UI and this kernel agree on the location without hardcoding a path.


In [ ]:
experiment_dir = os.environ.get("WEIGHTSLAB_ROOT_LOG_DIR") or os.path.abspath("./exp/mnist_mode_c")
os.makedirs(experiment_dir, exist_ok=True)
print("Experiment directory:", experiment_dir)

config = {
    "experiment_name": "mnist_mode_c",
    "device": str(device),
    "root_log_dir": experiment_dir,
    "num_classes": 10,
    "eval_full_to_train_steps_ratio": 500,
    "learning_rate": 0.001,
    "data": {
        "train_loader": {"batch_size": 128, "shuffle": True, "is_training": True,
                          "compute_hash": False, "preload_labels": True},
        "test_loader": {"batch_size": 256, "shuffle": False, "is_training": False,
                         "compute_hash": False, "preload_labels": True},
    },
}

# Register the config so the backend / UI read ports, TLS and root_log_dir from it.
wl.watch_or_edit(config, flag="hyperparameters", poll_interval=1.0)


## 3 · Model, data and tracked objects

Everything wrapped with `wl.watch_or_edit(...)` becomes visible and editable in the
UI. A dataset returning `(image, index, label)` gives every sample a stable identity
so per-sample signals line up across the run.


In [ ]:
class MNISTCustomDataset(Dataset):
    """MNIST that returns (image, index, label) so samples keep a stable identity."""

    def __init__(self, root, train=True, download=False, transform=None):
        self.mnist = datasets.MNIST(root=root, train=train, download=download, transform=None)
        self.transform = transform

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        if self.transform:
            image = self.transform(image)
        return image, idx, label


data_root = os.path.join(experiment_dir, "data")
os.makedirs(data_root, exist_ok=True)
train_ds = MNISTCustomDataset(root=data_root, train=True, download=True, transform=transforms.ToTensor())
test_ds = MNISTCustomDataset(root=data_root, train=False, download=True, transform=transforms.ToTensor())

model = wl.watch_or_edit(CNN().to(device), flag="model", device=device, compute_dependencies=True)
optimizer = wl.watch_or_edit(optim.Adam(model.parameters(), lr=config["learning_rate"]), flag="optimizer")

train_loader = wl.watch_or_edit(train_ds, flag="data", loader_name="train_loader", **config["data"]["train_loader"])
test_loader = wl.watch_or_edit(test_ds, flag="data", loader_name="test_loader", **config["data"]["test_loader"])

train_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"), flag="loss", signal_name="train-loss-CE", log=True)
test_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"), flag="loss", signal_name="test-loss-CE", log=True)
metric = wl.watch_or_edit(Accuracy(task="multiclass", num_classes=config["num_classes"]).to(device), flag="metric", signal_name="metric-ACC", log=True)


## 4 · Train / eval steps

In [ ]:
def train_step(loader, model, optimizer, criterion, device):
    with guard_training_context:
        inputs, ids, labels = next(loader)
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        preds = logits.argmax(dim=1, keepdim=True)
        loss = criterion(logits.float(), labels.long(), batch_ids=ids, preds=preds)
        loss.mean().backward()
        optimizer.step()
    return loss.mean().detach().cpu().item()


def evaluate(loader, model, criterion, metric, device):
    total = torch.tensor(0.0, device=device)
    n = len(loader)
    for inputs, ids, labels in loader:
        with guard_testing_context:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1, keepdim=True)
            total += criterion(logits, labels, batch_ids=ids, preds=preds).mean()
            metric.update(logits, labels)
    return (total / max(n, 1)).item(), (metric.compute() * 100).item()


## 5 · Serve the backend (this kernel)

Non-blocking: the gRPC server starts on background threads. This kernel is now the
backend the UI connects to.


In [ ]:
wl.serve(serving_grpc=True)


## 6 · Launch the UI (separate terminal)

```bash
weightslab start ./exp/mnist_mode_c
```

It opens `http://localhost:50051`. If you launched it *before* this notebook it sat
on a landing page and connects the moment `wl.serve(...)` above ran — the UI is fully
independent of this backend's lifecycle.


## 7 · Train in the background (notebook stays interactive)

The loop runs in a daemon thread, so this cell returns immediately and you can keep
running cells here, inspect in the UI, or open the in-browser notebook — all while
training streams live. Pause/resume from the UI, or stop the loop with `stop_event.set()`.


In [ ]:
stop_event = threading.Event()

def run_training(steps=8000):
    wl.start_training(timeout=3)              # flip the experiment into the training state
    eval_ratio = config["eval_full_to_train_steps_ratio"]
    pbar = tqdm(range(steps), desc="Training")
    for step in pbar:
        if stop_event.is_set():
            break
        age = model.get_age() if hasattr(model, "get_age") else step
        train_loss = train_step(train_loader, model, optimizer, train_criterion, device)
        postfix = {"loss": f"{train_loss:.3f}"}
        if age > 0 and age % eval_ratio == 0:
            _, test_acc = evaluate(test_loader, model, test_criterion, metric, device)
            postfix["test_acc"] = f"{test_acc:.1f}%"
        pbar.set_postfix(postfix)
    wl.write_history()
    wl.write_dataframe()
    print("Training loop finished. Logs at:", experiment_dir)

training_thread = threading.Thread(target=run_training, daemon=True, name="wl-notebook-training")
training_thread.start()
print("Training started in the background — keep using the notebook and the UI.")


## 8 · Interact while it trains

This kernel holds the live objects, so you can inspect them here at any time — the
same objects the UI is showing. For example:


In [ ]:
# Peek at live state from the same kernel (safe to run repeatedly while training).
print("model age (steps):", model.get_age() if hasattr(model, "get_age") else "n/a")
print("training thread alive:", training_thread.is_alive())

# To stop the background loop:
# stop_event.set()


## 9 · Curate in the UI

While the run streams, use Weights Studio to find the samples that matter (sort by
`train-loss-CE`, histogram it), tag the informative extremes, discard the redundant
middle, and resume on the curated set — no code changes, no restart. You can also
open the **in-browser notebook** (button left of the logo) for quick, backend-side
inspection with `df`, `model`, `cm`, `logger`, `hp` pre-bound.
